# Clean Survey Data

## Code Status

Complete - pending review

## Purpose

Prepare survey data for analysis regarding research Q’s

1.  Create dependent categorical variables:

- Largest apprehension between group discussions and solo conversations
- Most frequently used transportation between rideshare and public
  transit

1.  Basic cleaning of independent open-text variables for initial
    analysis:

- Remove punctuation and capitalization

## Setup

In [ ]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Rows: 273 Columns: 27
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (22): ResponseId, Q0, Q1, Q2, Q3, Q4, Q5, Q6, Q13, Q14, Q15, Q16, Q17, Q...
dbl  (5): StartDate, EndDate, Duration (in seconds), LocationLatitude, Locat...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.

Rows: 262 Columns: 11
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (7): Submission id, Participant id, Total approvals, Sex, Country of res...
dbl (4): Started at, Completed at, Time taken, Age

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.

## Missing data

For each research Q, we will be restricted to participants that answered
the corresponding open-text question.

In [ ]:
survey |> summarise(
  n_apprehension = sum(!is.na(q_advice_text)),
  n_transportation = sum(!is.na(q_transport_pref_text))
  )

# A tibble: 1 × 2
  n_apprehension n_transportation
           <int>            <int>
1            165              165

This restricts our initial sample size to n = 165 for both research
questions.

In [ ]:
# Missing data for those who provided q_advice_text
survey |>
  filter(!is.na(q_advice_text)) |>
  summarise(across(everything(), \(x) sum(is.na(x)))) |>
  pivot_longer(everything(), names_to = "col", values_to = "n_na") |>
  filter(n_na > 0) |>
  arrange(desc(n_na))

# A tibble: 19 × 2
   col                         n_na
   <chr>                      <int>
 1 Submission id                 12
 2 Started at                    12
 3 Completed at                  12
 4 Time taken                    12
 5 dem_age                       12
 6 Total approvals               12
 7 dem_sex                       12
 8 dem_country                   12
 9 dem_student                   12
10 dem_employment                12
11 Q0                            10
12 q29_rideshare_dayuse           7
13 q2_comfort_groupdiscussion     1
14 q13_nervous_newppl_convo       1
15 q14_nofear_speaking_convo      1
16 q17_relaxed_newppl_convo       1
17 q27_public_transit_dayuse      1
18 q20_license                    1
19 q21_car_access                 1

# A tibble: 19 × 2
   col                         n_na
   <chr>                      <int>
 1 Submission id                 12
 2 Started at                    12
 3 Completed at                  12
 4 Time taken                    12
 5 dem_age                       12
 6 Total approvals               12
 7 dem_sex                       12
 8 dem_country                   12
 9 dem_student                   12
10 dem_employment                12
11 Q0                            10
12 q29_rideshare_dayuse           7
13 q2_comfort_groupdiscussion     1
14 q13_nervous_newppl_convo       1
15 q14_nofear_speaking_convo      1
16 q17_relaxed_newppl_convo       1
17 q27_public_transit_dayuse      1
18 q20_license                    1
19 q21_car_access                 1

# A tibble: 2 × 2
   n_na     n
  <int> <int>
1     1     4
2     0   161

q_29_rideshare_dayuse is critical for engineering our outcome variable
for the transportation methods research question. The 7 NA observations
will need to be excluded, however those participants can be kept for the
apprehension research question.

There are also 4 observations in the apprehension research question
sample that are missing a value from 1 PRCA question each. While this is
a minimal amount of missing data, it is predominantly missing from solo
conversation Q’s, and may bias our engineered outcome of ‘biggest_fear’
as a result. These 4 observations will be removed from the apprehension
sample.

Removals for both samples will occur after all processing steps before
data is exported.

## Processing for apprehension research Q

In [ ]:
#| label: "apprehension_engineering"
# Identify forward and reverse-coded Q's (Higher scores = more apprehensive)
forward_code <- c(
  "q1_dislike_groupdiscussion",
  "q3_nervous_groupdiscussion",
  "q5_tense_newppl_groupdiscussion",
  "q13_nervous_newppl_convo",
  "q15_nervous_convo",
  "q18_afraid_convo"
  )
reverse_code <- c(
  "q2_comfort_groupdiscussion",
  "q4_like_groupdiscussion",
  "q6_relaxed_groupdiscussion",
  "q14_nofear_speaking_convo",
  "q16_relaxed_convo",
  "q17_relaxed_newppl_convo"
  )
# Set numeric values for Likert scales
likert_forward <- c(
  "Strongly disagree"          = 1,
  "Somewhat disagree"          = 2,
  "Neither agree nor disagree" = 3,
  "Somewhat agree"             = 4,
  "Strongly agree"             = 5
)
likert_reverse <- c(
  "Strongly disagree"          = 5,
  "Somewhat disagree"          = 4,
  "Neither agree nor disagree" = 3,
  "Somewhat agree"             = 2,
  "Strongly agree"             = 1
)
# Convert to numeric
survey <- survey |>
  mutate(
    across(all_of(forward_code), ~ likert_forward[.x]),
    across(all_of(reverse_code), ~ likert_reverse[.x])
  )
# Identify group discussion vs. solo conversation Q's
group_discussion <- c(
  "q1_dislike_groupdiscussion",
  "q2_comfort_groupdiscussion",
  "q3_nervous_groupdiscussion",
  "q4_like_groupdiscussion",
  "q6_relaxed_groupdiscussion"
  )
solo_conversation <- c(
  "q13_nervous_newppl_convo",
  "q14_nofear_speaking_convo",
  "q15_nervous_convo",
  "q16_relaxed_convo",
  "q17_relaxed_newppl_convo",
  "q18_afraid_convo"
  )
# Calculate biggest fear
survey <- survey |> 
  mutate(
    
#   Average of group apprehension Q's
    group_apprehension = rowMeans(
      pick(all_of(group_discussion)), na.rm = TRUE),
    
#   Average of solo apprehension Q's
    solo_apprehension = rowMeans(
      pick(all_of(solo_conversation)), na.rm = TRUE),

#   Assign greatest of the two values (equal becomes NA)
    biggest_fear = case_when(
      group_apprehension > solo_apprehension ~ "group_apprehension",
      solo_apprehension > group_apprehension ~ "solo_apprehension",
      .default = "equal")
  )

## Processing for transportation research Q

In [ ]:
# Set numeric values for transportation variables
usage_monthly <- c(
  "Never"                     = 0,
  "0-1 days a month"          = 1,
  "2-4 days a month"          = 3,
  "4-8 days a month"          = 6,
  "8 or more days a month"    = 8
  )
usage_daily <- c(
  "1-2 rides in a typical day"       = 1.5,
  "3-4 rides in a typical day"       = 3.5,
  "5-6 rides in a typical day"       = 5.5,
  "7 or more rides in a typical day" = 7
  )

# Calculate highest usage transport method
survey <- survey |>
  mutate(
    
#   Convert 3-month usage variables to numeric
    across(c(q26_public_transit_3months, q28_rideshare_3months), ~ usage_monthly[.x]),
    
#   Convert typical daily usage variables to numeric
    across(c(q27_public_transit_dayuse, q29_rideshare_dayuse), ~ usage_daily[.x]),

#   Multiply usage over past 3 months by typical daily usage (usage count estimate)
    public_transit_count = q26_public_transit_3months * q27_public_transit_dayuse,
    rideshare_count = q28_rideshare_3months * q29_rideshare_dayuse,

#   Assign highest usage transport method (equal if BA)
    highest_transport = case_when(
      public_transit_count > rideshare_count ~ "public_transit",
      rideshare_count > public_transit_count ~ "rideshare",
      public_transit_count == 0 & rideshare_count == 0 ~ "neither",
      .default = "equal" )
    )

## open text cleaning

This script will perform only basic cleaning of the text by removing
punctuation and capitalization. Further cleaning approaches such as
removing stop words or stemming/lemmatization will be considered in
other scripts.

In [ ]:
survey <- survey |> mutate(
    q_advice_text =
      str_to_lower(str_remove_all(q_advice_text, "[[:punct:]]")),
  q_transport_pref_text =
    str_to_lower(str_remove_all(q_transport_pref_text, "[[:punct:]]"))
  )

## Filter and export

In [ ]:
# Full cleaned data for EDA
survey |> write_csv(here::here(path_clean, "survey_clean.csv"))

survey |> filter(
  
# Remove observations with NA values
  !is.na(q_advice_text),
  if_all(q1_dislike_groupdiscussion:q18_afraid_convo, \(x) !is.na(x))) |>

# Remove redundant columns and save
  select(q_advice_text, biggest_fear) |>
  write_csv(here::here(path_clean, "survey_apprehension.csv"))

# Data for transport research Q
survey |> filter(
  
# Remove observations with NA values
  !is.na(q_transport_pref_text),
  if_all(q26_public_transit_3months:q29_rideshare_dayuse, \(x) !is.na(x))) |>
  
# Remove redundant columns and save
  select(q_transport_pref_text, highest_transport) |>
  write_csv(here::here(path_clean, "survey_transportation.csv"))